In [ ]:
!pip install deepface

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected
from IPython.display import display, Javascript
from google.colab.output import eval_js

# Function to ask the user to choose continue or finish the program by showing "Continue" and "Finish" buttons
#   [Parameters]
#     No parameter
#   [Returns]
#     If the user chose to finish the program, then this function returns "Yes". Otherwise, it returns "No".
def ask_if_finish():
  # Javascript code to show "Continue" and "Finish" buttons
  js = Javascript('''
    async function showButtons() {
      // Create the buttons
      const div = document.createElement('div');
      const btn_continue = document.createElement('button');
      btn_continue.textContent = 'Continue';
      const btn_finish = document.createElement('button');
      btn_finish.textContent = 'Finish';

      document.body.appendChild(div);
      div.appendChild(btn_continue);
      div.appendChild(btn_finish);

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      var is_finished;
      await new Promise((resolve) => {
        // If the user pushes "Continue" button, is_finished is set to false.
        btn_continue.onclick = () => { is_finished = false; resolve(); };
        // If the user pushes "Finish" button, is_finished is set to true.
        btn_finish.onclick = () => { is_finished = true; resolve(); };
      });

      // Remove all buttons after the user pushed one of the buttons.
      div.remove();

      return is_finished;
    }
  ''')
  # Read the above Javascript code (showButtons() is not executed at this time).
  display(js)
  # Evaluate the above Javascript code (showButtons() is executed through the evaluation).
  if eval_js('showButtons()'):
    return 'Yes'
  return 'No'


try:
  while True:
    # Take a photo.
    image_file = take_photo()

    try:
      # Detect a face and estimate emotions.
      emotion = DeepFace.analyze(img_path = image_file, actions = ['emotion'])[0]['emotion']

      # Print the probability of "Happiness"
      print('Probability of happiness is %f.' % emotion['happy'])

    except FaceNotDetected:
      # If no face is detected, a FaceNotDetected exception is raised.
      print('No face detected.')

    # If the user pushed the "Finish" button, then exit the program.
    if ask_if_finish() == 'Yes':
      break

except Exception as e:
  # If any other exceptions are raised, details will be displayed.
  print(e)